In [ ]:
from vla.models.smolvla import SmolVLAPolicy
import torch
from PIL import Image
from transformers import AutoProcessor


In [ ]:
checkpoint = "HuggingFaceVLA/smolvla_libero"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from checkpoint: {checkpoint} on device: {device}")
policy = SmolVLAPolicy(checkpoint=checkpoint, device=device, action_dim=7)

In [ ]:
image = Image.open("object_image.jpg").convert("RGB")
task = "what is in the image?"

In [ ]:
# # Load Image
# img_path = PROJECT_ROOT / "data/images/libero/object/ep0000_task_0/frame0000.png"
# image = Image.open(img_path).convert("RGB")

# # Load Full Task Description
# task_path = (
#     PROJECT_ROOT / "data/images/libero/object/ep0000_pick_up_the_orange_juice_and_place_it_in_the_basket/task.txt"
# )
# with open(task_path) as f:
#     task_description = f.read().strip()

# print(f"Task: {task_description}")

# # Pick a specific word to track (e.g., "bowl" or "box")
# target_word = "juice"

In [ ]:
# %% Setup: processor, hooks, and model references (done ONCE)
hf_vlm = policy.model.vlm_with_expert.vlm
llm = hf_vlm.model.text_model
total_layers = len(llm.layers)
layer_indices = list(range(total_layers // 2))  # SmolVLA discards final 50%
vision_dtype = next(hf_vlm.model.vision_model.parameters()).dtype

# Force eager attention output
llm.config._attn_implementation = "eager"
llm.config.output_attentions = True

# Load processor ONCE
try:
    processor = AutoProcessor.from_pretrained("HuggingFaceVLA/smolvla_libero")
except Exception:
    processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-500M-Instruct")

# Pre-compute the text prompt template (same for every frame)
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": task}]}]
prompt_template = processor.apply_chat_template(messages, add_generation_prompt=True)

print(f"Model ready — {total_layers} total layers, using first {len(layer_indices)}")
print(f"Processor loaded, prompt template cached.")


In [ ]:
# Format the prompt using the chat template (image first, then text)
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": task}]}]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

# Tokenize and move to device
inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

# Cast pixel_values to match the vision model dtype (bfloat16 on GPU)
vision_dtype = next(hf_vlm.model.vision_model.parameters()).dtype
inputs["pixel_values"] = inputs["pixel_values"].to(vision_dtype)

# Generate a response
with torch.no_grad():
    output_ids = hf_vlm.generate(**inputs, max_new_tokens=100)

# Decode only the newly generated tokens
generated_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
response = processor.decode(generated_tokens, skip_special_tokens=True)
print("VLM response:", response)
